# Грунт против «масложора»: 20 задач bAbI

Два участника на одних и тех же задачах и одном и том же официальном тесте (20 000 вопросов):

- **ученик** (`acquire.py` + `core.py`): общие схемы рассуждения без единого слова, язык задач выучивает сам по рассказам с ответами. Считает на процессоре;
- **обычный трансформер** с нуля (`colab/baseline.py`): тот же текст на входе, ответ на выходе. Считает на видеокарте.

Как запускать:

1. Среда выполнения → Сменить среду → T4 (или L4) и «Большой объём ОЗУ» (там больше процессоров).
2. Среда выполнения → Выполнить всё. Всё вместе идёт около полутора-двух часов: трансформер учится в фоне, пока считает ученик.
3. Если оборвалось, выполнить всё снова: продолжит с места остановки.
4. Нужны две таблицы между линиями из `=` (их печатает последняя ячейка).


In [ ]:
import os, subprocess, sys
WORK = '/content/mx-mind'
try:
    from google.colab import drive
    drive.mount('/content/drive')
    WORK = '/content/drive/MyDrive/mx-mind'
except Exception as e:
    print('Диск не подключён, итог останется в', WORK, '-', e)
os.makedirs(WORK, exist_ok=True)
if os.path.exists('/content/MX-Project'):
    subprocess.run(['git', '-C', '/content/MX-Project', 'pull', '-q'])
else:
    subprocess.run(['git', 'clone', '-q', '--depth', '1', 'https://github.com/kireikosasha/MX-Project', '/content/MX-Project'], check=True)
M5 = '/content/MX-Project/research_m5'
!python -u {M5}/colab/mind.py --work {WORK} --fetch


## Трансформер (в фоне)

Три объёма данных: те же 20 рассказов на задачу, стандартный набор (1000 вопросов на задачу) и большой (10 000 вопросов на задачу). Ход обучения пишется в файл `baseline.txt` в рабочей папке.

In [ ]:
base = subprocess.Popen([sys.executable, '-u', M5 + '/colab/baseline.py', '--data', WORK + '/tasks_1-20_v1-2', '--lang', 'en',
                         '--sizes', '20,1k,10k', '--out', WORK + '/baseline_en.json'],
                        stdout=open(WORK + '/baseline.txt', 'w'), stderr=subprocess.STDOUT)
print('трансформер запущен в фоне')


## Ученик

Английский, хинди и английский с заменёнными буквами (шифр) по 20 рассказов на задачу; для английского ещё 5, 10 и 50 рассказов. Раз в пять минут печатает, сколько задач пройдено.

In [ ]:
!python -u {M5}/colab/mind.py --work {WORK}


## Итог

In [ ]:
try:
    base.wait()
except NameError:
    pass
print('=' * 70)
print(open(WORK + '/results.md').read())
print('=' * 70)
lines = open(WORK + '/baseline.txt').read().splitlines()
print('\n'.join(l for l in lines if l.startswith('|')) or '\n'.join(lines[-15:]))
print('=' * 70)
